# 04 · Final out-of-sample evaluation

**Objective:** Evaluate the **frozen** model selected in notebook 03
against constant-return baselines on the previously untouched test
observations (March 2024 through August 2026).

**Primary metric:** RMSE. **Secondary metric:** MAE. Hyperparameters
and the evaluation rule are not changed using test results.


## 1. Restore the original time split

The shared helper implements exactly the same boundaries and one-row
target-protection gaps as notebook 03. The model will train on the
original training rows only; validation was used for selection, not
added to training afterward.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent
sys.path.insert(0, str(project_dir))

from src.qqq_utils import (
    FEATURE_COLUMNS, TARGET_COLUMN, SELECTED_RF_PARAMS,
    chronological_split, load_model_data, regression_metrics,
    selected_random_forest,
)

model_data = load_model_data(project_dir)
train_data, val_data, test_data = chronological_split(model_data)
print("Training rows:", len(train_data))
print("Validation rows:", len(val_data))
print("Test rows:", len(test_data))
print("Test period:", test_data.index.min().date(), "to",
      test_data.index.max().date())


Training rows: 2918
Validation rows: 624
Test rows: 626
Test period: 2024-03-04 to 2026-08-31


## 2. Refit only the selected model

The Random Forest parameters came from **validation** performance in
notebook 03: `n_estimators=200`, `max_depth=3`,
`min_samples_leaf=10`, `random_state=42`. We recreate rather than
retune the model, and fit using the same original training set.


In [2]:
X_train, y_train = train_data[FEATURE_COLUMNS], train_data[TARGET_COLUMN]
X_test, y_test = test_data[FEATURE_COLUMNS], test_data[TARGET_COLUMN]

final_rf_model = selected_random_forest().fit(X_train, y_train)
print("Frozen parameters:", SELECTED_RF_PARAMS)


Frozen parameters: {'n_estimators': 200, 'max_depth': 3, 'min_samples_leaf': 10, 'random_state': 42, 'n_jobs': -1}


## 3. Test predictions and baseline comparison

Always predicting zero or the *training* mean requires no learned
feature relationship. The first time we inspect test labels is here.
Both error metrics measure percentage-point forecasting errors.


In [3]:
training_mean = y_train.mean()
test_predictions_by_model = {
    "Predict 0%": np.zeros(len(y_test)),
    "Predict training mean": np.full(len(y_test), training_mean),
    "Tuned Random Forest": final_rf_model.predict(X_test),
}

test_performance = pd.DataFrame([
    {"Model": name, "Test MAE": regression_metrics(y_test, preds)["MAE"],
     "Test RMSE": regression_metrics(y_test, preds)["RMSE"]}
    for name, preds in test_predictions_by_model.items()
]).sort_values("Test RMSE").reset_index(drop=True)
test_performance.round(6)


,Model,Test MAE,Test RMSE
0,Tuned Random Forest,0.934910,1.314286
1,Predict training mean,0.937274,1.340726
2,Predict 0%,0.945665,1.343452


## 4. Are forecasts informative or just conservative?

Comparing the mean, standard deviation, and correlation of forecasts
versus realized returns reveals whether the model predicts much less
variation than QQQ actually experiences. Correlation alone does not
measure trading profitability.


In [4]:
rf_test_preds = test_predictions_by_model["Tuned Random Forest"]
prediction_diagnostics = pd.Series({
    "Mean forecast (%)": rf_test_preds.mean(),
    "Mean actual (%)": y_test.mean(),
    "Forecast std. dev. (percentage points)": rf_test_preds.std(),
    "Actual std. dev. (percentage points)": y_test.std(),
    "Prediction/actual correlation": np.corrcoef(rf_test_preds, y_test)[0, 1],
})
prediction_diagnostics


Mean forecast (%)                         0.101635
Mean actual (%)                           0.085555
Forecast std. dev. (percentage points)    0.205591
Actual std. dev. (percentage points)      1.341798
Prediction/actual correlation             0.204477
dtype: float64

## 5. Which features did the forest use?

Tree impurity-based importance allocates a score across predictors.
It is **not** a causal attribution and correlated predictors can share
importance in misleading ways.


In [5]:
feature_importance = pd.Series(
    final_rf_model.feature_importances_, index=FEATURE_COLUMNS,
    name="Impurity-based importance",
).sort_values(ascending=False)
feature_importance


MA_Distance_20d_Pct    0.410694
Momentum_5d_Pct        0.211714
Daily_Return_Pct       0.205710
Volatility_20d_Pct     0.171882
Name: Impurity-based importance, dtype: float64

## 6. Findings and limitations

**Recorded test RMSE (percentage points):** tuned Random Forest
`1.314286`; training-mean baseline `1.340726`; zero baseline `1.343452`.
The model reduced RMSE by approximately **2.17% relative to the zero
baseline**, a modest improvement. Its forecast/actual correlation was
about `0.2045`, with much smaller predicted than actual variation.

This single historical holdout is not a statistical significance test.
The study uses one ETF, four price-derived features, one validation
period, and a modest tuning grid. Because we have now seen the test
results, any subsequent changes must be labeled exploratory or
evaluated on genuinely new, untouched observations.

**Next:** Notebook 05 asks the different question of whether these
forecasts support a realistic trading rule with transaction costs.
